# 🏛️ Agent Constitution — Interactive Demo

**Governance review with adversarial debate, epistemic honesty, and retrospective verification primitives.**

> No API key needed. This notebook runs entirely with MockAdapter.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentPolis/agent-constitution/blob/main/examples/agent_constitution_demo.ipynb)

In [ ]:
# Install agent-constitution (takes ~10 seconds)
!pip install -q agent-constitution 2>/dev/null || pip install -q -e .. 2>/dev/null || print('Install from git instead:')
!pip install -q git+https://github.com/AgentPolis/agent-constitution.git 2>/dev/null || True

## 1. Create Agents with Constitutional Rules

Each agent has a `SOUL.md` defining its identity, values, and hard constraints.
The Constitution injects shared epistemic rules into every agent.

In [ ]:
from adapters import MockAdapter
from constitution import BaseAgent, Constitution, Debate

rules = Constitution.default()
adapter = MockAdapter(simulate_delay_ms=0)

analyst = BaseAgent(role="analyst", goal="Evaluate opportunities", adapter=adapter, constitution=rules)
critic = BaseAgent(role="critic", goal="Challenge assumptions", adapter=adapter, constitution=rules)
judge = BaseAgent(role="judge", goal="Render fair verdict", adapter=adapter, constitution=rules)

print("✅ Agents created with constitutional governance")
print(f"   Constitution: {len(rules.text)} chars of epistemic rules")
print("   Agents: analyst, critic, judge")

## 2. Run an Adversarial Debate

When an analyst gives a high score (≥ 32/40), a structured debate is triggered:
1. **Challenger** raises 3 specific challenges
2. **Defender** rebuts each challenge with evidence
3. **Judge** renders a verdict with score adjustment

In [ ]:
topic = "Should we expand from mid-market to enterprise this year?"

debate = Debate(challenger=critic, defender=analyst, judge=judge)
result = debate.run(topic)

print(f"📋 Topic: {topic}")
print("")
print(f"⚔️  Verdict: {result.verdict}")
print(f"📊 Score delta: {result.score_delta:+d}")
print("")
print(f"Challenges raised: {len(result.challenges)}")
for i, c in enumerate(result.challenges, 1):
    print(f"  {i}. {c[:100]}..." if len(c) > 100 else f"  {i}. {c}")
print("")
print(f"Defenses: {len(result.defenses)}")
for i, d in enumerate(result.defenses, 1):
    print(f"  {i}. {d[:100]}..." if len(d) > 100 else f"  {i}. {d}")

## 3. Retrospective Calibration

Track predictions over time. Did the analyst's optimism hold up? Did the critic's risks materialize?

In [ ]:
from constitution import Retrospective

retro = Retrospective()

# Record predictions
p1 = retro.record_prediction("analyst", "Enterprise expansion will improve win rates within two quarters", confidence=0.75)
p2 = retro.record_prediction("critic", "A direct enterprise push will stall without solutions engineering support", confidence=0.60)
p3 = retro.record_prediction("analyst", "Sales efficiency will remain healthy during the transition", confidence=0.55)

# Simulate verification (in production, this happens days/weeks later)
retro.verify(p1.id, "correct")    # Win rates improved
retro.verify(p2.id, "correct")    # Solutions engineering became the bottleneck
retro.verify(p3.id, "incorrect")  # Sales efficiency dipped during the transition

summary = retro.summary()
print("📊 Retrospective Summary")
print(f"   Predictions: {summary['total_predictions']}")
print(f"   Verified:    {summary['verified']}")
print(f"   Accuracy:    {summary['accuracy']:.0%}")
print("")
print("   Agent Credibility:")
for role, cred in summary['agent_credibility'].items():
    print(f"     {role}: {cred:.2f}")

## 4. Governance Score

Quantify how well your system follows constitutional governance.

In [ ]:
from constitution import compute_governance_score

report = compute_governance_score(
    debate_coverage=0.85,       # 85% of material evaluations triggered debate
    constitution_compliance=0.92, # 92% of responses passed compliance check
    calibration_accuracy=0.67,  # 67% retrospective accuracy
)

badge_emoji = {"green": "🟢", "yellow": "🟡", "red": "🔴", "uncalibrated": "⚪"}

print(f"{badge_emoji[report.badge]} Governance Score: {report.score}/100 — {report.label}")
print("")
print(f"   Debate rigor:            {report.debate_rigor:.0%}")
print(f"   Constitution compliance: {report.constitution_compliance:.0%}")
print(f"   Calibration accuracy:    {report.calibration_accuracy:.0%}")

## 5. Signal Pool — Cross-Source Validation

When multiple independent sources mention the same entity, confidence increases.

In [ ]:
from datetime import datetime

from constitution import Signal, SignalPool

pool = SignalPool()

pool.ingest(Signal(tier="T1", direction="bullish", title="Enterprise pilots asking for annual contracts",
    summary="Sales review shows multiple pilots asking for annual terms", source="sales_review", confidence=0.9, timestamp=datetime.now()))
pool.ingest(Signal(tier="T2", direction="bullish", title="Enterprise pilots asking for annual contracts",
    summary="Customer success notes confirm procurement interest", source="csm_notes", confidence=0.7, timestamp=datetime.now()))
pool.ingest(Signal(tier="T4", direction="bearish", title="Security review flagged incomplete audit-log coverage",
    summary="Compliance tracker shows launch blocker on audit logs", source="compliance_tracker", confidence=0.4, timestamp=datetime.now()))

# Cross-reference: same entity from 2+ sources
cross = pool.cross_reference()
actionable = pool.get_actionable(min_confidence=0.6)

print(f"📡 Signal Pool: {len(pool.signals)} signals")
print(f"   Cross-validated: {len(cross)}")
print(f"   Actionable (≥0.6 confidence): {len(actionable)}")
for s in actionable:
    print(f"     • [{s.tier}] {s.title} ({s.direction}, confidence={s.confidence})")

---

## Next Steps

- **Use real LLMs**: Set `ANTHROPIC_API_KEY` and use `AnthropicAPIAdapter`
- **Run locally**: Use `OllamaAdapter` with any open model (Llama 3, Mistral, etc.)
- **CLI**: `ac debate "your topic"` for instant debates from terminal
- **Docs**: [github.com/AgentPolis/agent-constitution](https://github.com/AgentPolis/agent-constitution)

**Agent Constitution** — a governance harness for high-stakes decisions.